In [ ]:
import os
from pathlib import Path

if not Path("predict.py").exists():
    !git clone https://github.com/KP-365/Fake_news
    os.chdir("Fake_news")

!python3 -m pip install -q -r requirements.txt

In [ ]:
from predict import ID_TO_LABEL, MAX_LENGTH, load_model

model, tokenizer, device = load_model()
print(f"Checkpoint loaded on {device}")

In [ ]:
import os
import re

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

dataset_path = kagglehub.dataset_download(
    "saurabhshahane/fake-news-classification"
)
welfake_df = pd.read_csv(os.path.join(dataset_path, "WELFake_Dataset.csv"))

if "Unnamed: 0" in welfake_df.columns:
    welfake_df = welfake_df.drop(columns=["Unnamed: 0"])

welfake_df["title"] = welfake_df["title"].fillna("")
welfake_df = welfake_df.dropna(subset=["text"])

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    return re.sub(r"\s+", " ", text).strip()

welfake_df["title"] = welfake_df["title"].apply(clean_text)
welfake_df["text"] = welfake_df["text"].apply(clean_text)
welfake_df = welfake_df[welfake_df["text"].str.len() > 0]
welfake_df["content"] = (
    welfake_df["title"] + ". " + welfake_df["text"]
).str.strip(". ")
welfake_df = welfake_df.drop_duplicates(subset=["text"])

train_df, temp_df = train_test_split(
    welfake_df,
    test_size=0.3,
    stratify=welfake_df["label"],
    random_state=42,
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42,
)

print(f"Held-out test examples: {len(test_df):,}")
print(test_df["label"].value_counts().sort_index())

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
)

batch_size = 32
test_texts = test_df["content"].tolist()
true_labels = test_df["label"].to_numpy()
predicted_labels = []
prediction_confidences = []

model.eval()
for start in range(0, len(test_texts), batch_size):
    batch_texts = test_texts[start:start + batch_size]
    encoded = tokenizer(
        batch_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}

    with torch.no_grad():
        probabilities = torch.softmax(model(**encoded).logits, dim=-1)

    confidences, predictions = probabilities.max(dim=-1)
    predicted_labels.extend(predictions.cpu().tolist())
    prediction_confidences.extend(confidences.cpu().tolist())

predicted_labels = np.asarray(predicted_labels)
prediction_confidences = np.asarray(prediction_confidences)
accuracy = accuracy_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels, pos_label=1)
matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 ({ID_TO_LABEL[1]} class): {f1:.4f}")
print("Confusion matrix:")
print(matrix)
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=[ID_TO_LABEL[0], ID_TO_LABEL[1]],
).plot(cmap="Blues")

In [ ]:
sample_positions = np.linspace(0, len(test_df) - 1, 5, dtype=int)
sample_predictions = test_df.iloc[sample_positions][["title", "label"]].copy()
sample_predictions["true_label"] = [
    ID_TO_LABEL[label] for label in sample_predictions["label"]
]
sample_predictions["predicted_label"] = [
    ID_TO_LABEL[label] for label in predicted_labels[sample_positions]
]
sample_predictions["confidence"] = prediction_confidences[sample_positions]
sample_predictions.drop(columns=["label"]).reset_index(drop=True)